# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **UCI Cleveland Heart Disease Dataset**.
Dataset ini memuat data klinis pasien untuk memprediksi keberadaan penyakit jantung (binary classification).
Dataset terdiri dari 303 data pasien dengan 14 kolom:
- **age**: Usia pasien
- **sex**: Jenis kelamin (1 = laki-laki, 0 = perempuan)
- **cp**: Tipe nyeri dada (0-3)
- **trestbps**: Tekanan darah istirahat (mm Hg)
- **chol**: Kolesterol serum (mg/dl)
- **fbs**: Gula darah puasa > 120 mg/dl (1 = true, 0 = false)
- **restecg**: Hasil elektrokardiografi istirahat (0-2)
- **thalach**: Detak jantung maksimum yang dicapai
- **exang**: Angina akibat olahraga (1 = ya, 0 = tidak)
- **oldpeak**: Depresi ST yang diinduksi oleh olahraga relatif terhadap istirahat
- **slope**: Kemiringan segmen ST puncak latihan (0-2)
- **ca**: Jumlah pembuluh darah utama (0-4)
- **thal**: Jenis kelainan thalasemia (0-3)
- **target**: Status penyakit jantung (1 = sakit, 0 = sehat)

# **2. Import Library**

Pada tahap ini, kita mengimpor pustaka Python yang dibutuhkan untuk manipulasi data, analisis statistik, visualisasi, dan prapemrosesan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import os

# **3. Memuat Dataset**

Memuat dataset dari folder `heart_disease_raw` dan memeriksa baris pertama data serta informasi tipe data kolom.

In [ ]:
raw_csv_path = "../heart_disease_raw/heart-disease.csv"
df = pd.read_csv(raw_csv_path)
print("Baris Pertama Dataset:")
print(df.head())
print("\nInformasi Dataset:")
df.info()

# **4. Exploratory Data Analysis (EDA)**

Melakukan EDA untuk memahami karakteristik dataset, memeriksa statistik deskriptif, korelasi antar fitur, duplikasi data, dan memeriksa apakah terdapat missing values.

In [ ]:
print(f"Bentuk dataset: {df.shape}")
print("\nDeskripsi Statistik:")
print(df.describe())

print("\nMengecek Missing Values:")
print(df.isnull().sum())

print("\nMengecek Duplikasi Data:")
print(f"Jumlah data duplikat: {df.duplicated().sum()}")

# Korelasi antar fitur
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Matriks Korelasi Fitur")
plt.show()

# Distribusi Target
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df, palette='Set2')
plt.title("Distribusi Target (0 = Healthy, 1 = Heart Disease)")
plt.show()

# **5. Data Preprocessing**

Melakukan prapemrosesan data agar siap digunakan untuk model machine learning:
1. Menghapus data duplikat.
2. Deteksi dan penanganan outlier menggunakan clipping berbasis IQR pada fitur numerik kontinu.
3. Standarisasi fitur numerik dengan `StandardScaler`.
4. Melakukan encoding One-Hot pada fitur kategorikal (`cp`, `restecg`, `slope`, `ca`, `thal`).
5. Menyimpan data preprocessed.

In [ ]:
# 1. Menghapus duplikasi
initial_len = len(df)
df.drop_duplicates(inplace=True)
print(f"Menghapus {initial_len - len(df)} baris duplikat.")

# 2. Penanganan Outlier dengan IQR Clipping
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    df[col] = np.clip(df[col], lower_bound, upper_bound)
print("Outliers di-clip menggunakan IQR.")

# 3. Standarisasi Fitur Numerik
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])
print("Fitur numerik telah distandarisasi.")

# 4. Encoding Kategorikal (One-Hot Encoding)
cat_cols = ['cp', 'restecg', 'slope', 'ca', 'thal']
for col in cat_cols:
    df[col] = df[col].astype(str)
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Mengonversi nilai dummy True/False menjadi 1/0 integer
dummy_cols = [c for c in df_encoded.columns if c not in num_cols + ['sex', 'fbs', 'exang', 'target']]
for c in dummy_cols:
    df_encoded[c] = df_encoded[c].astype(int)

for c in ['sex', 'fbs', 'exang', 'target']:
    if c in df_encoded.columns:
        df_encoded[c] = df_encoded[c].astype(int)

# 5. Menyimpan data
os.makedirs("heart_disease_preprocessing", exist_ok=True)
output_path = "heart_disease_preprocessing/heart-disease-preprocessed.csv"
df_encoded.to_csv(output_path, index=False)
print(f"Preprocessed data disimpan di {output_path} dengan bentuk: {df_encoded.shape}")